# 基于 CEDAR 的乙酰氯间歇精馏过程时序因果分析：复现 Notebook

本 Notebook 复现论文中的全部数值结果与插图：

1. 读取 4 s 采样的 DCS 记录（27 个测点、24 345 行）；
2. 复算描述统计、运行阶段划分与一步自相关；
3. 读取 CEDAR 计算结果 `cedar_4s.joblib`，重建 75 条边表并做拓扑分级；
4. 生成论文 Fig. 1–Fig. 6 的 300 dpi PNG 与矢量 PDF。

> CEDAR 推断本身不在本 Notebook 内重跑（需 GPU 与数小时机时），
> 直接复用随包分发的计算结果对象；其余步骤均为确定性计算。

## 0 环境与路径

依赖：`numpy`、`pandas`、`scipy`、`matplotlib`、`joblib`。
若需重跑 CEDAR，额外安装 `causalts==0.28.0` 与 `torch`。

In [ ]:
from pathlib import Path
import sys, runpy, json

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("工作目录:", ROOT)

## 1 数据与记录口径

`data/process_4s.csv` 为分析用数据表：首列为时间戳，其余 27 列为匿名测点。
记录间隔恒为 4 s，覆盖 2026-09-16 04:25:22 至 2026-09-17 07:28:18，共 24 345 行。

In [ ]:
df = pd.read_csv(ROOT / "data/process_4s.csv", parse_dates=["timestamp"])
dt = df.timestamp.diff().dt.total_seconds().dropna()

print("形状:", df.shape)
print("时间范围:", df.timestamp.iloc[0], "→", df.timestamp.iloc[-1])
print("步长唯一值:", sorted(dt.unique())[:5], "…")
print("变量数:", df.shape[1] - 1)
df.head(3)

## 2 定量诊断：统计、阶段与自相关

`src/analysis.py` 依次完成：变量描述统计、按时间六分段的运行剖面、
一步自相关以及 48 条跨测点边的拓扑分级，
结果写入 `outputs/`。

In [ ]:
runpy.run_path(str(ROOT / "src/analysis.py"), run_name="__main__")

In [ ]:
stats = json.loads((ROOT / "outputs/stats.json").read_text(encoding="utf-8"))
stats

关键读数（与论文正文一致）：

- 相邻记录等值比例 37%–99%，一步自相关中位数 1.000、最小值 0.952，
  说明记录经保持/平滑处理，强自相关是必须写入限制的硬约束；
- 过程非线性，本文不设线性、秩相关或互信息基线，也不报告任何相关系数；
- 48 条跨测点边中，同单元 3 条、顺流相邻 9 条、逆流相邻 6 条、跨单元远端 30 条。

In [ ]:
acf = pd.read_csv(ROOT / "outputs/autocorr_lag1.csv")

print("ACF(1) 中位数 %.3f，最小 %.3f" % (acf.acf_lag1.median(), acf.acf_lag1.min()))
acf.sort_values("acf_lag1").head(8)

## 3 CEDAR 结果：直接读取作者提供的计算对象

`cedar_4s.joblib` 为 `causalts.cedar.result.CedarResult`。
约定：`cg_tig[i, j, k] == 1` 表示变量 *i* 在滞后 *k* 步上指向变量 *j*。
本数据 lag0 全为 0，lag1 共 75 条边（27 条自回归 + 48 条跨测点）。

In [ ]:
import joblib

res = joblib.load(ROOT / "data/cedar_4s.joblib")
names = list(res.var_names)
cg = np.asarray(res.cg_tig)
val = np.asarray(res.val_matrix)

print("变量数:", len(names))
print("cg_tig 形状:", cg.shape, "（滞后维 = 0…%d）" % (cg.shape[2] - 1))
print("lag0 边数:", int((cg[:, :, 0] != 0).sum()), " lag1 边数:", int((cg[:, :, 1] != 0).sum()))
print("max_lag =", res.max_lag, "| alpha_cond1 =", res.alpha_cond1,
      "| alpha_cond2 =", res.alpha_cond2, "| pruned =", res.pruned)
print("条件独立检验次数 n_tests =", res.n_tests)
print("滞后选择:", res.lag_method, "| p 值方法:", res.lag_pvalue_method)

In [ ]:
rows = []
for k in range(cg.shape[2]):
    idx = np.argwhere(cg[:, :, k] != 0)
    for i, j in idx:
        rows.append({"src": names[i], "dst": names[j], "lag": k,
                     "val": float(val[i, j, k])})
edges = pd.DataFrame(rows)
edges["auto"] = [s.split("-")[0] == d.split("-")[0] and s == d for s, d in
                 zip(edges.src, edges.dst)]
edges["auto"] = [s == d for s, d in zip(edges.src, edges.dst)]

dist = pd.read_csv(ROOT / "data/cedar_edges_4s.csv")
merged = edges.merge(dist, on=["src", "dst", "lag"], suffixes=("", "_ref"))
assert len(merged) == len(dist), "边表与分发结果不一致"
print("边总数 %d（自回归 %d、跨测点 %d）"
      % (len(edges), int(edges.auto.sum()), int((~edges.auto).sum()))
print("与分发边表逐条一致:", np.allclose(merged.val, merged.val_ref))
edges.head(10)

## 4 拓扑分级与强度排序

按图 1 的设备连接关系，把 48 条跨测点边分为同单元、顺流相邻、逆流相邻、
跨单元远端四类，并按 |边强度| 降序排列（`outputs/edges_48_classified.csv`）。

In [ ]:
e48 = pd.read_csv(ROOT / "outputs/edges_48_classified.csv")
print(e48["拓扑关系"].value_counts().to_string())
print("\n跨测点图密度 = 48/(27×26) = %.2f%%" % (48 / (27 * 26) * 100))
e48.head(15)[["src", "dst", "强度", "拓扑关系"]]

## 5 工艺预期对照（表 4、表 6）

`data/expectations_27.csv` 是论文表 4 的机器可读版本：27 条预期关系在对照之前即按工艺资料定好等级与方向，
`result` 列由 `outputs/edges_48_classified.csv` 自动填出，不做事后调整。

本单元复算三件事：

1. 条目数、A/B 分级与同向/反向/未检出的计数，须与正文 14.8% / 7.4% / 77.8% 一致；
2. 每条“同向检出”的记录必须在 48 条边表中真实存在，每条“反向”记录必须是正向缺失、反向存在；
3. 表 6 的分级分解（A 级 3/0/15，B 级 1/2/6）。


In [ ]:
exp = pd.read_csv(ROOT / "data/expectations_27.csv")
edges = pd.read_csv(ROOT / "outputs/edges_48_classified.csv")
have = {(s, t) for s, t in zip(edges["src"], edges["dst"])}

print(exp[["id", "src", "dst", "class", "result"]].to_string(index=False))
print()

# 1) 计数一致性
n = len(exp)
same = int((exp["result"] == "同向检出").sum())
rev = int((exp["result"] == "反向").sum())
miss = int((exp["result"] == "未检出").sum())
assert n == 27 and (exp["class"] == "A").sum() == 18 and (exp["class"] == "B").sum() == 9
assert (same, rev, miss) == (4, 2, 21), (same, rev, miss)

# 2) 与边表逐条互证
for r in exp.itertuples():
    if r.result == "同向检出":
        assert (r.src, r.dst) in have, r.id
    elif r.result == "反向":
        assert (r.dst, r.src) in have and (r.src, r.dst) not in have, r.id
    else:
        assert (r.src, r.dst) not in have and (r.dst, r.src) not in have, r.id

# 3) 表 6 分级分解
print(f"{'class':<6}{'n':>4}{'same':>7}{'reversed':>10}{'not retained':>15}")
for cls in ("A", "B"):
    sub = exp[exp["class"] == cls]
    print(f"{cls:<6}{len(sub):>4}"
          f"{int((sub['result'] == '同向检出').sum()):>7}"
          f"{int((sub['result'] == '反向').sum()):>10}"
          f"{int((sub['result'] == '未检出').sum()):>15}")
print(f"{'total':<6}{n:>4}{same:>7}{rev:>10}{miss:>15}")
print()
print("同向命中：", ", ".join(exp.loc[exp.result == '同向检出', 'matched_edge']))
print("强度尾端：0.000011（KTL-T1->KTL-P1）、0.000134（CND-TA2->RFD-L1）落在低强度尾部。")


In [ ]:
proto = pd.read_csv(ROOT / "data/expectations_protocol.csv")
print(proto[["expectation", "src", "dst", "cls", "skeleton", "direction", "strict_result"]].to_string(index=False))
print()

n = len(proto)
skel = int(proto["skeleton"].sum())
a = proto[proto["cls"] == "A"]
b = proto[proto["cls"] == "B"]
a_skel = int(a["skeleton"].sum())
b_skel = int(b["skeleton"].sum())
same = int((proto["direction"] == "same").sum())
rev = int((proto["direction"] == "reversed").sum())

assert (n, skel, a_skel, b_skel, same, rev) == (27, 6, 3, 3, 4, 2), (n, skel, a_skel, b_skel, same, rev)

print(f"严格定向召回（主结果）  {same}/{n} = {same/n:.1%}")
print(f"骨架召回（不计方向）    {skel}/{n} = {skel/n:.1%}")
print(f"A 级骨架                {a_skel}/{len(a)} = {a_skel/len(a):.1%}")
print(f"B 级骨架                {b_skel}/{len(b)} = {b_skel/len(b):.1%}")
print(f"骨架命中内定向一致率    {same}/{same+rev} = {same/(same+rev):.1%}")


## 6 工艺支持等级与排序顶端精度

对 48 条跨测点边逐条判定工艺支撑等级：`D` 直接（设备连接、能量路径或物料路径与方向一致），
`I` 间接（闭环控制、共享公用工程负荷或经一个中间单元的传播），`N` 无支持（既无路径也无控制理由）。
判据逐条写在 `data/edge_support_48.csv` 的 `basis_zh` 列。再按 |强度| 排序，统计前 k 条中带工艺依据的比例
（precision@k），这是工程复核实际使用的操作点。

In [ ]:
sup = pd.read_csv(ROOT / "data/edge_support_48.csv")
prec = pd.read_csv(ROOT / "data/precision_at_k.csv")

assert len(sup) == 48
cnt = sup["tier"].value_counts()
for t, label in (("D", "直接"), ("I", "间接或闭环"), ("N", "无支持")):
    n = int(cnt.get(t, 0))
    print("%s %2d 条（%.1f%%）" % (label, n, n / len(sup) * 100))

print()
print("k   有依据  支持率   直接依据")
for _, r in prec.iterrows():
    print("%-3d %-7s %-8s %s" % (int(r.k), "%d/%d" % (r.supported, r.k),
                                 "%.1f%%" % r.precision, "%d/%d" % (r.direct, r.k)))

p5 = prec.loc[prec.k == 5].iloc[0]
assert int(p5.supported) == 4, p5
assert abs(float(p5.precision) - 80.0) < 1e-6
print()
print("前 5 条候选 4 条有工艺依据，支持率 %.1f%%；前 3 条为 100%%。" % p5.precision)

## 7 生成 Fig. 1–Fig. 6

Fig. 1 为工艺与测点布置示意；Fig. 2–Fig. 6 分别对应批次运行剖面、
因果网络、强度排名、工艺预期对照与数据诊断。
图件输出到 `figures/`（PNG 300 dpi + PDF 矢量）。

In [ ]:
runpy.run_path(str(ROOT / "src/fig1_process.py"), run_name="__main__")
runpy.run_path(str(ROOT / "src/figures.py"), run_name="__main__")

In [ ]:
from IPython.display import Image, display

for name in ["Fig1_process_sensors", "Fig2_batch_profile", "Fig3_cedar_network",
             "Fig4_top15_strength", "Fig5_expectation_vs_detected",
             "Fig6_data_diagnostics"]:
    print(name)
    display(Image(filename=str(ROOT / "figures" / (name + ".png")), width=900))

## 8 复现边界

- **记录间隔 4 s、单批次单窗口**：max_lag = 1 仅覆盖 4 s 滞后，物理滞后更长的传递
  只能经由中间变量间接体现；结论的适用边界应写入论文讨论。
- **逐对检验未做网络级校正**（α_cond1 = α_cond2 = 0.01）：1323 次检验下
  期望假阳性约 13 条量级，48 条边中低强度尾部应谨慎解释。
- **无真值图**：本文以工艺知识构造 27 条预期关系做弱监督对照，
  其中仅 4 条同向检出，该结果本身即限制的一部分。
- **数据经保持/平滑处理**：相邻等值率 37%–99%、ACF(1) ≈ 1，
  表明有效独立样本量远小于 24 345，置信区间宽度被系统性低估。